# Yerbanalytics — Baseline de clasificación (MobileNetV3 + Transfer Learning)

Detección de anomalías en plantines de yerba mate. Este notebook entrena un
**baseline** con **transfer learning** sobre **MobileNetV3-Large** (TensorFlow/Keras),
usando datasets donantes curados (té, café) como sustituto de yerba real.

**Clases (las que hoy tienen donante):**

| Clase | Donante |
|---|---|
| `Sano` | Tea Leaf Dataset — Healthy · RoCoLe — healthy |
| `Clorosis` | CoLeaf — deficiencias N/Fe/Mg/Mn |
| `Dano_biotico` | Tea Leaf Dataset — 5 enfermedades · **RoCoLe — ácaro + roya (fondo real)** |
| `Estres_solar` | Tea Leaf Diseases — Sunlight Scorching |

> **Recordá:** los donantes son sólo para *training*. La validación final se hace
> contra **yerba real de Misiones**. Hay *domain gap* (fondo controlado vs. plantín
> en vivero) — el transfer learning lo tolera, pero el test honesto es con yerba.

## 1. Montar Google Drive

Los datasets viven en tu Drive (no en el repo). Montamos para acceder a ellos.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuración

Todo lo que vas a tocar está acá. Si renombraste algo en tu Drive, ajustá las rutas.

- `IMG_SIZE`: MobileNetV3 trabaja bien en 224x224.
- `SAMPLES_PER_CLASS`: submuestreamos para **equilibrar**. Clorosis es el cuello de
  botella (~291), así que apuntamos cerca. **Mejor pocas parejas que un desbalance brutal.**
- `USE_ROCOLE`: RoCoLe etiqueta por **CSV** (no por carpetas). Si está en `True`, lo
  leemos y sumamos su **ácaro de café con fondo de campo real** a `Dano_biotico`.

In [ ]:
import os, random, shutil, json
from pathlib import Path
import numpy as np
import tensorflow as tf

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

DRIVE_DATASETS = Path('/content/drive/MyDrive/Personal/Yerbanalytics/datasets')
UNIFIED_DIR = Path('/content/dataset_unified')

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SAMPLES_PER_CLASS = 300
VAL_SPLIT = 0.2
EPOCHS_HEAD = 12
EPOCHS_FINETUNE = 8

CLASSES = ['Sano', 'Clorosis', 'Dano_biotico', 'Estres_solar']

# Donantes que YA vienen en carpetas (Forma A: carpeta = etiqueta)
SOURCES = {
    'Sano': [
        'Tea Leaf Dataset/Healthy Leaves/Healthy_leaves',
    ],
    'Clorosis': [
        'CoLeaf/iron-Fe', 'CoLeaf/magnesium-Mg',
        'CoLeaf/manganese-Mn', 'CoLeaf/nitrogen-N',
    ],
    'Dano_biotico': [
        'Tea Leaf Dataset/Diseased Leaves/Blister_Blight',
        'Tea Leaf Dataset/Diseased Leaves/Brown_Blight',
        'Tea Leaf Dataset/Diseased Leaves/Leaf_Red_Rust',
        'Tea Leaf Dataset/Diseased Leaves/Red_Spider_Mite',
        'Tea Leaf Dataset/Diseased Leaves/Tea_Mosquito_Bug',
    ],
    'Estres_solar': [
        'Tea Leaf Diseases Dataset Towards Accurate Field Diagnosis Using Image-Based Detection/Research Dataset/Raw Dataset/Sunlight Scorching',
    ],
}

# --- RoCoLe (Forma B: etiquetas en CSV) ---
USE_ROCOLE = True
ROCOLE_DIR = DRIVE_DATASETS / 'RoCoLe A robusta coffee leaf images dataset'
# que clase de RoCoLe mapea a cada clase nuestra
ROCOLE_MAP = {
    'red_spider_mite': 'Dano_biotico',   # acaro de cafe — fondo de campo real (valor unico)
    'rust_level_1': 'Dano_biotico',
    'rust_level_2': 'Dano_biotico',
    'rust_level_3': 'Dano_biotico',
    'rust_level_4': 'Dano_biotico',
    'healthy': 'Sano',
}

print('TensorFlow:', tf.__version__)
print('GPU disponible:', tf.config.list_physical_devices('GPU'))

## 3. Armar el dataset unificado por clase

Traducimos los donantes a **NUESTRAS clases**. Los que vienen en carpetas se leen directo;
**RoCoLe se resuelve leyendo su CSV** (`Annotations/RoCoLE-csv.csv`): la etiqueta está
embebida en la columna `Label` como JSON, y el nombre de archivo en `External ID`.
Después mezclamos y tomamos hasta `SAMPLES_PER_CLASS` para equilibrar.

In [ ]:
IMG_EXT = {'.jpg', '.jpeg', '.png'}

def list_images(folder: Path):
    if not folder.exists():
        print(f'  AVISO: no existe {folder}')
        return []
    return [p for p in folder.rglob('*') if p.suffix.lower() in IMG_EXT]

def rocole_by_class():
    """Lee el CSV de RoCoLe y devuelve {clase_nuestra: [rutas]}."""
    import pandas as pd
    csv_path = ROCOLE_DIR / 'Annotations' / 'RoCoLE-csv.csv'
    photos = ROCOLE_DIR / 'Photos'
    if not csv_path.exists():
        print(f'  AVISO: no encuentro el CSV de RoCoLe en {csv_path}')
        return {}
    df = pd.read_csv(csv_path)
    out = {}
    for _, row in df.iterrows():
        try:
            cls = json.loads(row['Label']).get('classification')
        except Exception:
            cls = None
        target = ROCOLE_MAP.get(cls)
        if target is None:
            continue
        img = photos / str(row['External ID'])
        if img.exists():
            out.setdefault(target, []).append(img)
    return out

# Reset de la carpeta unificada
if UNIFIED_DIR.exists():
    shutil.rmtree(UNIFIED_DIR)

rocole = rocole_by_class() if USE_ROCOLE else {}
if rocole:
    print('RoCoLe aporto:', {k: len(v) for k, v in rocole.items()})

resumen = {}
for clase in CLASSES:
    candidatos = []
    for sub in SOURCES[clase]:
        candidatos += list_images(DRIVE_DATASETS / sub)
    candidatos += rocole.get(clase, [])      # sumar RoCoLe (Forma B)
    random.shuffle(candidatos)
    elegidas = candidatos[:SAMPLES_PER_CLASS]

    destino = UNIFIED_DIR / clase
    destino.mkdir(parents=True, exist_ok=True)
    for i, src in enumerate(elegidas):
        shutil.copy(src, destino / f'{clase}_{i:04d}{src.suffix.lower()}')
    resumen[clase] = (len(candidatos), len(elegidas))

print('\nclase            disponibles  usadas')
for c, (disp, usa) in resumen.items():
    print(f'{c:16} {disp:>10}  {usa:>5}')

## 4. Cargar como `tf.data` (split train / validation)

`image_dataset_from_directory` infiere las etiquetas del nombre de carpeta y arma el
split. `label_mode='int'` → etiquetas enteras (usamos `SparseCategoricalCrossentropy`).
`prefetch` solapa carga y cómputo para que la GPU no espere.

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    UNIFIED_DIR, validation_split=VAL_SPLIT, subset='training', seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='int')

val_ds = tf.keras.utils.image_dataset_from_directory(
    UNIFIED_DIR, validation_split=VAL_SPLIT, subset='validation', seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='int')

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)
print('Clases (en orden):', class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

## 5. Sanity check — ver un batch

Antes de entrenar, **mirá los datos con tus ojos**. Si las etiquetas no cuadran con
las imágenes, no hay modelo que valga.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 9))
for images, labels in train_ds.take(1):
    for i in range(min(9, images.shape[0])):
        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names[labels[i].numpy()])
        plt.axis('off')
plt.tight_layout(); plt.show()

## 6. Data augmentation

Generamos variantes (flip, rotación, zoom) para que el modelo no memorice y generalice
mejor — clave porque tenemos **pocas imágenes por clase**. Sólo se aplica en training.

> **Gotcha de MobileNetV3:** en Keras el modelo **ya incluye el preprocessing**
> (normalización) cuando `include_preprocessing=True` (por defecto). Por eso le pasamos
> las imágenes en rango **[0, 255]** tal cual — **no** hay que reescalar a mano.

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
], name='data_augmentation')

## 7. Construir el modelo (MobileNetV3-Large)

**Transfer learning:** tomamos MobileNetV3 ya entrenado en ImageNet (sabe ver bordes,
texturas, formas), le **congelamos** el cuerpo (`trainable = False`) y le ponemos una
**cabeza nueva** para nuestras clases. Así reusamos lo aprendido y entrenamos poquito.

In [ ]:
base_model = tf.keras.applications.MobileNetV3Large(
    input_shape=IMG_SIZE + (3,), include_top=False,
    weights='imagenet', include_preprocessing=True)
base_model.trainable = False   # fase 1: cuerpo congelado

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)
model.summary()

## 8. Compilar y entrenar — Fase 1 (cabeza)

`EarlyStopping` corta si la validación deja de mejorar y restaura los mejores pesos.
`ModelCheckpoint` guarda el mejor modelo en el Drive por las dudas.

> Si el desbalance siguiera molestando, acá podrías pasar `class_weight=...` a `.fit()`.
> Como ya submuestreamos para equilibrar, arrancamos sin eso.

In [ ]:
CKPT_DIR = Path('/content/drive/MyDrive/Personal/Yerbanalytics/modelos')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4,
                                     restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(str(CKPT_DIR / 'best_head.keras'),
                                       monitor='val_accuracy', save_best_only=True),
]

hist_head = model.fit(train_ds, validation_data=val_ds,
                      epochs=EPOCHS_HEAD, callbacks=callbacks)

## 9. Fine-tuning — Fase 2 (descongelar capas superiores)

Ahora descongelamos las **capas de arriba** del cuerpo (las que aprenden rasgos
específicos) y reentrenamos con un **learning rate bajito**, para ajustar sin destruir
lo aprendido en ImageNet. Las capas de abajo (rasgos genéricos) las dejamos quietas.

In [ ]:
base_model.trainable = True
FINE_TUNE_AT = len(base_model.layers) - 40
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss=tf.keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

hist_ft = model.fit(train_ds, validation_data=val_ds,
                    epochs=EPOCHS_FINETUNE, callbacks=callbacks)

## 10. Curvas de entrenamiento

Si la curva de validación se separa mucho de la de training → **overfitting**
(memorizó en vez de aprender). Con pocas imágenes es el riesgo número uno.

In [ ]:
def juntar(h1, h2, key):
    return h1.history[key] + h2.history[key]

acc = juntar(hist_head, hist_ft, 'accuracy')
val_acc = juntar(hist_head, hist_ft, 'val_accuracy')
loss = juntar(hist_head, hist_ft, 'loss')
val_loss = juntar(hist_head, hist_ft, 'val_loss')

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(acc, label='train'); plt.plot(val_acc, label='val')
plt.title('Accuracy'); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(loss, label='train'); plt.plot(val_loss, label='val')
plt.title('Loss'); plt.legend()
plt.show()

## 11. Evaluación — matriz de confusión y reporte

La accuracy sola engaña. La **matriz de confusión** te dice *qué* clases se confunden
entre sí. Mirá especialmente si **Clorosis** se mezcla con **Estres_solar** o con
**Dano_biotico** (amarilleos/necrosis parecidos) — ese era el riesgo que veníamos marcando.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import itertools

y_true, y_pred = [], []
for images, labels in val_ds:
    probs = model.predict(images, verbose=0)
    y_pred += list(np.argmax(probs, axis=1))
    y_true += list(labels.numpy())

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap='Blues')
plt.title('Matriz de confusion'); plt.colorbar()
ticks = range(NUM_CLASSES)
plt.xticks(ticks, class_names, rotation=45, ha='right'); plt.yticks(ticks, class_names)
for i, j in itertools.product(ticks, ticks):
    plt.text(j, i, cm[i, j], ha='center',
             color='white' if cm[i, j] > cm.max() / 2 else 'black')
plt.ylabel('Real'); plt.xlabel('Prediccion'); plt.tight_layout(); plt.show()

## 12. Guardar el modelo y exportar a TFLite (edge)

Guardamos el modelo entrenado y lo convertimos a **TFLite cuantizado** — el formato
liviano para correr en el **edge** (el gantry). La cuantización baja el tamaño y acelera
la inferencia con una pérdida mínima de precisión. **Ésta era la ventaja de TF que charlamos.**

In [ ]:
model.save(str(CKPT_DIR / 'yerbanalytics_baseline.keras'))

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_path = CKPT_DIR / 'yerbanalytics_baseline.tflite'
tflite_path.write_bytes(tflite_model)
print('Guardado:', tflite_path, f'({len(tflite_model)/1e6:.2f} MB)')

## Próximos pasos / notas

- **Curar a ojo** las clases ruidosas (ácaro/scorching donde el síntoma no se ve) antes
  de subir `SAMPLES_PER_CLASS`.
- RoCoLe ya está integrado vía CSV (`USE_ROCOLE = True`). Si querés sólo el ácaro y no la
  roya, dejá en `ROCOLE_MAP` únicamente `red_spider_mite`.
- **Validar contra yerba real** de Misiones cuando esté disponible (julio).
- Probar `class_weight` o *focal loss* si reaparece el desbalance.
- Una vez sólido el baseline de 4 clases, evaluar la 5ta clase del MVP (sin donante aún).

## 13. Smoke test — yerba real de Misiones (cualitativo)

Cargamos el modelo entrenado y lo corremos sobre las **pocas fotos de yerba real** que
llegaron de Misiones. **Esto NO es un test honesto ni una métrica:** son ~7 imágenes de
**una sola clase** (`Dano_biotico`: ácaro + daño biótico), sin poder estadístico.

El objetivo es **mirar con los ojos** si el modelo —entrenado con té y café de fondo
controlado— transfiere algo al **dominio yerba** (fondo de campo, hoja coriácea real):

- Si las manda a `Dano_biotico` con confianza razonable → el transfer aguanta el domain gap.
- Si las dispersa con confianza alta → confirma que aprendió el dataset donante, no el síntoma.
- La foto con **lupa** está fuera de distribución (el modelo nunca vio una) — no es veredicto.

> **Requisito:** subí la carpeta `imagenes-de-misiones/` a tu Drive en
> `datasets/imagenes-de-misiones/` (con las subcarpetas `acaro/` y `d-biotico/`).


In [ ]:
# === Smoke test cualitativo: yerba real de Misiones ===
# OJO: NO es una metrica. Son ~7 fotos de UNA sola clase real (Dano_biotico).
# Es para MIRAR con los ojos si el modelo transfiere al dominio yerba.

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from pathlib import Path

# --- Config (self-contained: corre aunque reinicies el runtime) ---
IMG_SIZE = (224, 224)
# Orden alfabetico de las clases (asi las ordena image_dataset_from_directory)
CLASS_NAMES = ['Clorosis', 'Dano_biotico', 'Estres_solar', 'Sano']

CKPT_DIR = Path('/content/drive/MyDrive/Personal/Yerbanalytics/modelos')
MISIONES_DIR = Path('/content/drive/MyDrive/Personal/Yerbanalytics/datasets/imagenes-de-misiones')

# Carpeta de Misiones -> clase esperada del modelo (acaro y d-biotico = Dano_biotico)
CARPETA_A_CLASE = {'acaro': 'Dano_biotico', 'd-biotico': 'Dano_biotico'}

# --- 1. Ver que hay en la carpeta de modelos y elegir el archivo ---
print('Modelos en', CKPT_DIR, ':')
for p in sorted(CKPT_DIR.glob('*.keras')):
    print('  -', p.name)

MODEL_PATH = CKPT_DIR / 'best_head.keras'   # <- cambia el nombre aca si queres otro
print('\nCargando:', MODEL_PATH.name)
model = tf.keras.models.load_model(str(MODEL_PATH))

# --- 2. Juntar las fotos de Misiones (con su clase esperada) ---
IMG_EXT = {'.jpg', '.jpeg', '.png'}
muestras = []  # (ruta, clase_esperada)
for carpeta, clase in CARPETA_A_CLASE.items():
    for img in sorted((MISIONES_DIR / carpeta).glob('*')):
        if img.suffix.lower() in IMG_EXT:
            muestras.append((img, clase))
print(f'\n{len(muestras)} imagenes de yerba real encontradas.')

# --- 3. Inferencia + grilla visual ---
# MobileNetV3 ya incluye el preprocessing -> le pasamos [0,255] tal cual.
n = len(muestras)
cols = 3
rows = (n + cols - 1) // cols
plt.figure(figsize=(cols * 4, rows * 4))

print('\narchivo                 esperado       prediccion     conf')
print('-' * 62)
for i, (ruta, esperada) in enumerate(muestras):
    img = tf.keras.utils.load_img(ruta, target_size=IMG_SIZE)
    arr = tf.keras.utils.img_to_array(img)              # [0,255]
    probs = model.predict(arr[None, ...], verbose=0)[0]
    idx = int(np.argmax(probs))
    pred = CLASS_NAMES[idx]
    conf = float(probs[idx])
    ok = (pred == esperada)

    print(f'{ruta.parent.name}/{ruta.name:14} {esperada:14} '
          f'{pred:14} {conf:5.1%}  {"OK" if ok else "X"}')

    ax = plt.subplot(rows, cols, i + 1)
    ax.imshow(np.array(img).astype('uint8'))
    color = 'green' if ok else 'red'
    ax.set_title(f'{ruta.parent.name}/{ruta.name}\n-> {pred} ({conf:.0%})',
                 color=color, fontsize=10)
    ax.axis('off')

plt.tight_layout(); plt.show()

# --- 4. Recordatorio honesto ---
print('\nNOTA: esto NO es accuracy. Con ~7 imgs de una sola clase no hay poder')
print('estadistico. Mira la confianza y a donde van los errores con tus ojos.')
print('La foto con LUPA es medio fuera de distribucion: no la tomes como veredicto.')